# RSNA Knee — Preprocessing v4 (laterality canonicalization)

**Kenapa v4 (20 Agustus 2026):** riset ulang Code/Discussion/eksternal menemukan kernel publik
`sadamtorres/domain-adaptation-beats-resolution-dinov2-on-knee` (LB 0,883) yang menjelaskan bug
struktural yang belum kita tangani: **4 dari 12 label (Medial/Lateral Meniscus, Medial/Lateral OA)
bisa belajar dari anatomi CERMIN-TERBALIK separuh waktu kalau lateralitas (kiri/kanan lutut) tidak
dikanonikalisasi** -- "the model sees mirrored anatomy half the time... nothing crashes, the loss
still goes down, and you lose a third of the score without a symptom." Tag `Laterality` kosong di
~50,7% studi kita (EDA sebelumnya) -- v1/v2/v3 semuanya TIDAK menangani ini sama sekali.

**Scope v4 (SATU perubahan besar, disiplin sama spt versi2 sebelumnya)**: kanonikalisasi SEMUA
studi seolah lutut KANAN --
1. **Deteksi lateralitas**: pakai tag `Laterality`/`ImageLaterality` kalau ada & baku (L/R). Kalau
   kosong/tidak baku, fallback ke **posisi geometris** -- hitung CENTER citra (bukan CORNER dari
   `ImagePositionPatient` langsung) di ruang pasien via `ImageOrientationPatient`+`PixelSpacing`+
   `Rows`/`Columns`, pakai tanda koordinat-x (konvensi DICOM LPS: x+ = kiri pasien) sbg sinyal.
   **DIVALIDASI DULU** (cell diagnostik) terhadap ~2000 series yang tag-nya ADA, sebelum dipakai
   produksi -- kalau agreement rendah, deteksi geometris JANGAN dipakai (fallback: tidak di-flip).
2. **Flip horizontal utk slot Coronal & Axial** dari studi lutut KIRI (bukan Sagittal -- 1 slice
   sagittal tidak punya konten kiri-kanan di dalamnya, flip di situ tidak menyelesaikan masalah
   medial/lateral). Arah flip (baris vs kolom) ditentukan dari komponen-x mana yang dominan di
   `row_cosine`/`col_cosine` (deteksi orientasi citra relatif sumbu pasien).
3. **SENGAJA BELUM dikerjakan**: pembalikan urutan slice sagittal per-lateralitas (dijelaskan
   sumber sbg penting utk arsitektur pooling-berbasis-posisi-slice mereka -- arsitektur kita
   pakai mean-pooling sederhana atas K_SLICE, urutan tidak berpengaruh, jadi item ini TIDAK
   krusial utk kita & butuh verifikasi geometris lebih hati-hati sebelum dikerjakan).

Urutan slice fisik (v2, `get_sorted_files`) TIDAK berubah, tetap dipakai. Kernel BARU (slug
terpisah dari v2/v3), cache v2 (dipakai checkpoint v15, model terbaik saat ini) tetap utuh.


## 0. Setup

In [ ]:
import os, sys, json, hashlib, traceback, warnings, random, time, re
from pathlib import Path
from collections import Counter, defaultdict
from multiprocessing import Pool

import numpy as np
import pandas as pd
from scipy import ndimage as ndi

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
RNG = np.random.default_rng(42)
random.seed(42)

T0 = time.time()
def tlog(msg):
    print(f"[t={time.time()-T0:7.1f}s] {msg}", flush=True)

def ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name); return True
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {pkg}")
        try:
            importlib.import_module(name); return True
        except Exception as e:
            print(f"[warn] gagal import/install {pkg}: {e}"); return False

HAVE_PYDICOM = ensure("pydicom")
HAVE_LANGDETECT = ensure("langdetect")
if HAVE_PYDICOM:
    import pydicom
if HAVE_LANGDETECT:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 42

OUT_DIR = Path("prep_outputs")
CACHE_DIR = OUT_DIR / "image_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
kaggle_input = Path("/kaggle/input")
CANDIDATE_ROOTS = [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                    Path("../data"), Path("data")]
if kaggle_input.exists():
    for pattern in ("*/train.csv", "*/test.csv", "*/*/train.csv", "*/*/test.csv"):
        for hit in kaggle_input.glob(pattern):
            CANDIDATE_ROOTS.insert(0, hit.parent)
DATA_DIR = next((p for p in CANDIDATE_ROOTS if p.exists() and ((p/"train.csv").exists() or (p/"test.csv").exists())), None) \
           or next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
tlog(f"DATA_DIR: {DATA_DIR} | exists: {DATA_DIR.exists()}")

train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
train = train.drop_duplicates(subset="StudyInstanceUID").reset_index(drop=True)
train_series = train_series.drop_duplicates(subset=["StudyInstanceUID", "SeriesInstanceUID"]).reset_index(drop=True)
tlog(f"train: {train.shape} | train_series: {train_series.shape}")

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
LABEL_COLS = [c for c in LABEL_COLS if c in train.columns]
gold_studies = set(train.loc[train[LABEL_COLS].notna().any(axis=1), "StudyInstanceUID"])
tlog(f"Studi gold-label: {len(gold_studies)}")

TRAIN_DIR = DATA_DIR / "train_series"

def list_dcm(study, series):
    d = TRAIN_DIR / study / series
    return sorted(d.glob("*.dcm")) if d.exists() else []


## 1. Investigasi tag `Laterality` + validasi deteksi geometris (belum pernah dicek sebelumnya)

Krusial untuk canonicalization: 4 label (Medial/Lateral Meniscus, Medial/Lateral OA) rusak diam2
kalau lutut kiri/kanan campur tanpa dikanonikalisasi. Tag kosong ~50% -- perlu fallback geometris,
DIVALIDASI dulu terhadap sampel yang tag-nya ADA sebelum dipercaya utk sampel yang tag-nya kosong.


In [ ]:
LAT_SAMPLE_N = min(2000, len(train_series))
lat_sample = train_series.sample(LAT_SAMPLE_N, random_state=17)
lat_values = Counter()
lat_fail = 0

for _, row in lat_sample.iterrows():
    files = list_dcm(row["StudyInstanceUID"], row["SeriesInstanceUID"])
    if not files:
        lat_fail += 1
        continue
    try:
        ds = pydicom.dcmread(files[0], force=True, stop_before_pixels=True)
        lat = getattr(ds, "Laterality", None)
        lat_img = getattr(ds, "ImageLaterality", None)
        lat_values[(str(lat), str(lat_img))] += 1
    except Exception:
        lat_fail += 1

tlog(f"Sampel diuji: {LAT_SAMPLE_N} series (gagal: {lat_fail})")
print("Distribusi (Laterality, ImageLaterality):")
for k, v in lat_values.most_common():
    print(f"  {k}: {v}")

LATERALITY_AVAILABLE = any(k[0] not in ("None", "") or k[1] not in ("None", "") for k in lat_values)
print(f"\n>>> Tag laterality {'TERSEDIA' if LATERALITY_AVAILABLE else 'TIDAK TERSEDIA'} di data ini.")


def compute_center_x(ds):
    """v4: hitung koordinat-x CENTER citra di ruang pasien (BUKAN corner dari ImagePositionPatient
    langsung) -- sinyal geometris cadangan utk lateralitas kalau tag kosong. Konvensi DICOM LPS:
    x+ = kiri pasien, x- = kanan pasien. Formula (sumber: kernel publik sadamtorres, diverifikasi
    ulang di bawah thd tag Laterality asli sebelum dipakai produksi):
      centre = IPP + row_cosine * col_spacing * (Columns/2) + col_cosine * row_spacing * (Rows/2)
    row_cosine = IOP[0:3] (arah pertambahan KOLOM), col_cosine = IOP[3:6] (arah pertambahan BARIS).
    """
    try:
        iop = [float(x) for x in ds.ImageOrientationPatient]
        row_cosine = np.array(iop[:3])
        col_cosine = np.array(iop[3:])
        ipp = np.array([float(x) for x in ds.ImagePositionPatient])
        ps = [float(x) for x in ds.PixelSpacing]  # [row_spacing, col_spacing]
        rows_n, cols_n = int(ds.Rows), int(ds.Columns)
        centre = ipp + row_cosine * ps[1] * (cols_n / 2.0) + col_cosine * ps[0] * (rows_n / 2.0)
        return float(centre[0]), row_cosine, col_cosine
    except Exception:
        return None, None, None


def flip_axis_from_orientation(row_cosine, col_cosine):
    """v4: tentukan axis citra mana yg paling sejajar sumbu-x pasien (kiri-kanan) -- dipakai utk
    tahu arah flip yg benar (baris vs kolom) saat kanonikalisasi. Kalau |row_cosine_x| dominan,
    sumbu KOLOM citra ~ sumbu-x pasien -> flip kiri-kanan = flip kolom (fliplr). Sebaliknya flip baris."""
    if abs(row_cosine[0]) >= abs(col_cosine[0]):
        return "columns"
    return "rows"


# --- VALIDASI: cocokkan deteksi geometris vs tag Laterality asli pada series yg tag-nya ADA ---
LAT_VALID_N = min(2000, len(train_series))
lat_valid_sample = train_series.sample(LAT_VALID_N, random_state=23)
agree_normal, agree_flipped, compared, no_tag, no_geom = 0, 0, 0, 0, 0
for _, row in lat_valid_sample.iterrows():
    files = list_dcm(row["StudyInstanceUID"], row["SeriesInstanceUID"])
    if not files:
        continue
    try:
        ds = pydicom.dcmread(files[0], force=True, stop_before_pixels=True)
    except Exception:
        continue
    tag = str(getattr(ds, "Laterality", None) or getattr(ds, "ImageLaterality", None) or "")
    tag_up = tag.upper()
    if not (tag_up.startswith("L") or tag_up.startswith("R")):
        no_tag += 1
        continue
    tag_side = "L" if tag_up.startswith("L") else "R"
    cx, _, _ = compute_center_x(ds)
    if cx is None:
        no_geom += 1
        continue
    compared += 1
    geom_side_normal = "L" if cx > 0 else "R"   # konvensi DICOM LPS standar: x+ = kiri
    geom_side_flipped = "R" if cx > 0 else "L"  # jaga-jaga kalau konvensi data ini terbalik
    if geom_side_normal == tag_side:
        agree_normal += 1
    if geom_side_flipped == tag_side:
        agree_flipped += 1

pct_normal = agree_normal / max(1, compared) * 100
pct_flipped = agree_flipped / max(1, compared) * 100
tlog(f"Validasi lateralitas geometris vs tag asli ({compared} series dibandingkan, "
     f"{no_tag} tag kosong/tdk baku dilewati, {no_geom} geometri gagal dihitung):")
tlog(f"  Konvensi normal (x+ = kiri): {agree_normal}/{compared} cocok ({pct_normal:.1f}%)")
tlog(f"  Konvensi terbalik (x+ = kanan): {agree_flipped}/{compared} cocok ({pct_flipped:.1f}%)")

# v4: pilih konvensi dgn agreement lebih tinggi, TAPI kalau keduanya rendah (<85%), JANGAN percaya
# sinyal geometris sama sekali -- lebih aman skip flip drpd flip salah arah (lebih buruk dari tidak
# di-flip sama sekali, krn salah arah = SEMUA studi tanpa tag jadi tercermin-balik, bukan cuma
# separuh spt v1-v3).
GEOM_LAT_TRUSTED = max(pct_normal, pct_flipped) >= 85.0
GEOM_LAT_FLIP_CONVENTION = pct_flipped > pct_normal  # True = pakai konvensi terbalik
tlog(f"GEOM_LAT_TRUSTED = {GEOM_LAT_TRUSTED} | GEOM_LAT_FLIP_CONVENTION = {GEOM_LAT_FLIP_CONVENTION}")
if not GEOM_LAT_TRUSTED:
    print(">>> AGREEMENT RENDAH (<85%) -- sinyal geometris TIDAK dipakai, studi tanpa tag Laterality")
    print(">>> TIDAK di-flip sama sekali (aman, sama spt perilaku v1-v3, bukan bahaya baru).")
else:
    print(f">>> Sinyal geometris terpercaya ({max(pct_normal, pct_flipped):.1f}%) -- dipakai sbg")
    print(">>> fallback lateralitas utk studi tanpa tag Laterality.")

def determine_laterality(ds):
    """v4: tag Laterality/ImageLaterality kalau baku, else fallback geometris (KALAU terpercaya
    dari validasi di atas). Return 'L', 'R', atau None (tidak bisa ditentukan -> tidak di-flip)."""
    tag = str(getattr(ds, "Laterality", None) or getattr(ds, "ImageLaterality", None) or "")
    tag_up = tag.upper()
    if tag_up.startswith("L"):
        return "L"
    if tag_up.startswith("R"):
        return "R"
    if not GEOM_LAT_TRUSTED:
        return None
    cx, _, _ = compute_center_x(ds)
    if cx is None:
        return None
    if GEOM_LAT_FLIP_CONVENTION:
        return "R" if cx > 0 else "L"
    return "L" if cx > 0 else "R"


## 2. Statistik normalisasi per grup sekuens

Dihitung dari sampel besar, dipakai konsisten oleh proses caching di bawah maupun notebook lain yang membaca cache ini.

In [ ]:
STATS_SAMPLE_N = min(3000, len(train_series))
stats_sample = train_series.sample(STATS_SAMPLE_N, random_state=29)
stats_rows = []

for _, row in stats_sample.iterrows():
    files = list_dcm(row["StudyInstanceUID"], row["SeriesInstanceUID"])
    if not files:
        continue
    try:
        ds = pydicom.dcmread(files[len(files) // 2], force=True)
        arr = ds.pixel_array.astype(np.float64)
        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
        arr = arr * slope + intercept
        group = f"{row['Anatomical_Plane']}|FS{row['Fluid_Sensitive']}|Fat{row['Fat_Suppression']}"
        stats_rows.append({"group": group, "p1": np.percentile(arr, 1), "p50": np.percentile(arr, 50),
                            "p99": np.percentile(arr, 99), "mean": arr.mean(), "std": arr.std()})
    except Exception:
        continue

stats_df = pd.DataFrame(stats_rows)
group_norm_stats = stats_df.groupby("group").agg(
    p1=("p1", "median"), p99=("p99", "median"), mean=("mean", "mean"), std=("std", "mean"),
    n_sampled=("group", "size")).reset_index()
group_norm_stats.to_csv(OUT_DIR / "group_norm_stats.csv", index=False)
tlog(f"Statistik grup dihitung dari {len(stats_df)} series sampel, {len(group_norm_stats)} grup unik.")
display(group_norm_stats)

GROUP_STATS = {r["group"]: (r["p1"], r["p99"]) for _, r in group_norm_stats.iterrows()}
FALLBACK_STATS = (float(stats_df["p1"].median()), float(stats_df["p99"].median()))


## 3. Pembersihan teks laporan (per studi)

In [ ]:
def clean_report(text):
    if not isinstance(text, str):
        return None
    t = re.sub(r"\[[A-Z_]+\]", " ", text)   # buang placeholder de-identifikasi mis. [DATE]
    t = re.sub(r"\s+", " ", t).strip()
    return t if t else None

def detect_lang_safe(text):
    if not isinstance(text, str) or len(text.strip()) < 8 or not HAVE_LANGDETECT:
        return None
    try:
        return detect(text)
    except Exception:
        return None

train["Report_clean"] = train["Report"].apply(clean_report)
train["language"] = train["Report_clean"].apply(detect_lang_safe)
train["has_gold_label"] = train["StudyInstanceUID"].isin(gold_studies)

study_manifest_cols = ["StudyInstanceUID", "Report_clean", "language", "has_gold_label"] + LABEL_COLS
study_manifest = train[study_manifest_cols].copy()
study_manifest.to_csv(OUT_DIR / "study_manifest.csv", index=False)
tlog(f"study_manifest.csv: {study_manifest.shape}")
print("Distribusi bahasa:")
print(study_manifest["language"].value_counts().head(12))


## 4. Fungsi worker: preprocessing 1 series penuh

In [ ]:
N_SLICES = 16
IMG_SIZE = 224
TARGET_MM_PER_PX = 0.5  # skala fisik seragam sebelum resize piksel

def quick_hash(arr_uint8):
    return hashlib.md5(arr_uint8.tobytes()).hexdigest()

def get_sorted_files(files):
    """v2: urutkan file DICOM 1 series berdasarkan POSISI FISIK, bukan nama file. Riset Discussion
    kompetisi (docs/RESEARCH_NOTES.md §15c) menemukan urutan nama file cuma benar ~5% dari waktu di
    dataset ini -- v1 diam-diam pakai urutan nama file (sorted(*.dcm)) apa adanya, artinya 16 slice
    "representatif" yang di-cache v1 kemungkinan besar BUKAN 16 titik yang tersebar rata sepanjang
    anatomi spt yang diasumsikan, tapi acak relatif terhadap anatomi.

    Posisi dihitung dari proyeksi ImagePositionPatient ke sumbu normal slice (cross product 2 arah
    ImageOrientationPatient) -- standar DICOM utk urutan slice yang benar terlepas dari skema
    penamaan file scanner apa pun. Fallback berjenjang kalau tag tidak ada/rusak: SliceLocation ->
    InstanceNumber -> urutan file asli (i, perilaku v1 dipertahankan sbg jaring pengaman).

    Return (files_terurut, n_fallback, pos_source) -- pos_source dicatat ke manifest utk diagnostik
    seberapa sering tiap sumber posisi terpakai (transparan, bukan diam-diam spt kelemahan v1)."""
    keyed = []
    n_fallback = 0
    sources = Counter()
    for i, f in enumerate(files):
        pos, src = None, None
        try:
            ds = pydicom.dcmread(f, force=True, stop_before_pixels=True)
            try:
                iop = [float(x) for x in ds.ImageOrientationPatient]
                row_dir, col_dir = np.array(iop[:3]), np.array(iop[3:])
                normal = np.cross(row_dir, col_dir)
                ipp = np.array([float(x) for x in ds.ImagePositionPatient])
                pos, src = float(np.dot(normal, ipp)), "ipp"
            except Exception:
                pass
            if pos is None:
                try:
                    pos, src = float(ds.SliceLocation), "slice_location"
                except Exception:
                    pass
            if pos is None:
                try:
                    pos, src = float(ds.InstanceNumber), "instance_number"
                except Exception:
                    pass
        except Exception:
            pass
        if pos is None:
            n_fallback += 1
            pos, src = float(i), "filename_order"  # fallback: perilaku v1 apa adanya
        sources[src] += 1
        keyed.append((pos, i, f))
    keyed.sort(key=lambda t: (t[0], t[1]))  # tiebreak stabil pakai urutan file asli
    pos_source = sources.most_common(1)[0][0] if sources else "filename_order"
    return [f for _, _, f in keyed], n_fallback, pos_source

def preprocess_slice(ds, group_key):
    arr = ds.pixel_array.astype(np.float64)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept

    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        arr = arr.max() - arr

    p1, p99 = GROUP_STATS.get(group_key, FALLBACK_STATS)
    arr = np.clip(arr, p1, p99)
    rng = (p99 - p1) if (p99 - p1) > 1e-6 else 1.0
    arr = (arr - p1) / rng  # -> [0,1] relatif terhadap grup

    ps = getattr(ds, "PixelSpacing", None)
    mm_per_px = float(ps[0]) if ps else TARGET_MM_PER_PX
    zoom_factor = mm_per_px / TARGET_MM_PER_PX
    if abs(zoom_factor - 1.0) > 0.02:
        arr = ndi.zoom(arr, zoom_factor, order=1)

    h, w = arr.shape[:2]
    if h != IMG_SIZE or w != IMG_SIZE:
        pad_h, pad_w = max(0, IMG_SIZE - h), max(0, IMG_SIZE - w)
        if pad_h or pad_w:
            arr = np.pad(arr, ((pad_h // 2, pad_h - pad_h // 2), (pad_w // 2, pad_w - pad_w // 2)), mode="constant")
        h, w = arr.shape[:2]
        top, left = max(0, (h - IMG_SIZE) // 2), max(0, (w - IMG_SIZE) // 2)
        arr = arr[top: top + IMG_SIZE, left: left + IMG_SIZE]
        if arr.shape != (IMG_SIZE, IMG_SIZE):  # jaga-jaga sisa pembulatan
            arr = np.pad(arr, ((0, IMG_SIZE - arr.shape[0]), (0, IMG_SIZE - arr.shape[1])), mode="constant")

    return np.clip(arr * 255, 0, 255).astype(np.uint8)

def process_one_series(args):
    study, series, plane, fs, fatsup = args
    files = list_dcm(study, series)
    if not files:
        return dict(study=study, series=series, status="MISSING_FOLDER", n_cached=0,
                     laterality=None, n_instances=0, error=None, pos_source=None, n_pos_fallback=0,
                     laterality_final=None, flipped=False)

    # v2: urutkan dulu berdasarkan posisi fisik SEBELUM pilih N_SLICES titik linspace -- supaya
    # 16 slice yang dipilih benar2 tersebar rata sepanjang anatomi, bukan sepanjang urutan file.
    files_sorted, n_pos_fallback, pos_source = get_sorted_files(files)

    group_key = f"{plane}|FS{fs}|Fat{fatsup}"
    idxs = np.linspace(0, len(files_sorted) - 1, min(N_SLICES, len(files_sorted))).astype(int)
    idxs = sorted(set(idxs.tolist()))

    slices, hashes = [], set()
    laterality = None
    laterality_final, flip_axis, need_flip = None, None, False
    try:
        for i, fi in enumerate(idxs):
            ds = pydicom.dcmread(files_sorted[fi], force=True)
            if i == 0:
                laterality = str(getattr(ds, "Laterality", None) or getattr(ds, "ImageLaterality", None) or "")
                # v4: kanonikalisasi lateralitas -- HANYA utk slot Coronal/Axial (1 slice sagittal
                # tidak punya konten kiri-kanan di dalamnya, flip di situ tidak menyelesaikan apa2).
                # Kanonikal ke "seolah lutut KANAN" -- studi KIRI di-flip horizontal.
                if plane in ("Coronal", "Axial"):
                    laterality_final = determine_laterality(ds)
                    if laterality_final == "L":
                        need_flip = True
                        _, row_cosine, col_cosine = compute_center_x(ds)
                        if row_cosine is not None:
                            flip_axis = flip_axis_from_orientation(row_cosine, col_cosine)
            arr8 = preprocess_slice(ds, group_key)
            if need_flip and flip_axis is not None:
                arr8 = np.fliplr(arr8) if flip_axis == "columns" else np.flipud(arr8)
            h = quick_hash(arr8)
            if h in hashes:   # exact-duplicate slice -> lewati, jangan dobel-hitung sinyal sama
                continue
            hashes.add(h)
            slices.append(arr8)

        if not slices:
            return dict(study=study, series=series, status="EMPTY_AFTER_DEDUP", n_cached=0,
                         laterality=laterality, n_instances=len(files), error=None,
                         pos_source=pos_source, n_pos_fallback=n_pos_fallback,
                         laterality_final=laterality_final, flipped=need_flip)

        stack = np.stack(slices, axis=0)
        out_dir = CACHE_DIR / study
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{series}.npz"
        np.savez_compressed(out_path, slices=stack)
        return dict(study=study, series=series, status="OK", n_cached=len(slices),
                     laterality=laterality, n_instances=len(files), error=None,
                     cache_path=str(out_path), bytes=out_path.stat().st_size,
                     pos_source=pos_source, n_pos_fallback=n_pos_fallback,
                     laterality_final=laterality_final, flipped=need_flip)
    except Exception as e:
        return dict(study=study, series=series, status="ERROR", n_cached=0,
                     laterality=laterality, n_instances=len(files), error=f"{e}\n{traceback.format_exc()}",
                     pos_source=pos_source, n_pos_fallback=n_pos_fallback,
                     laterality_final=laterality_final, flipped=need_flip)


## 5. Loop utama: paralel, prioritas studi gold-label, checkpoint resumable, dual safety cutoff

In [ ]:
N_WORKERS = 4
BATCH_SIZE = 150
SAFETY_CUTOFF_SECONDS = 7 * 3600
SAFETY_CUTOFF_BYTES = 16 * 1024**3   # 16 GB -- jaga margin dari kuota /kaggle/working ~21GB
DEBUG_LIMIT_SERIES = None  # v4 RUN PENUH -- sanity sukses: agreement geometris 97.4% (>85% ambang),
                           # semua slot Coronal/Axial di sampel berhasil ditentukan lateralitasnya.

# resume: cek manifest sebelumnya (dataset ter-attach) kalau ada
prior_manifest_candidates = [Path("/kaggle/working/prior_series_manifest.csv"), Path("prior_series_manifest.csv")]
if kaggle_input.exists():
    for hit in kaggle_input.glob("*/series_manifest.csv"):
        prior_manifest_candidates.insert(0, hit)
prior_manifest_path = next((p for p in prior_manifest_candidates if p.exists()), None)

done_keys = set()
MANIFEST_PATH = OUT_DIR / "series_manifest.csv"
MANIFEST_COLS = ["StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression",
                  "Laterality", "n_instances_total", "n_slices_cached", "status", "cache_path", "bytes",
                  "pos_source", "n_pos_fallback",  # v2: diagnostik urutan slice fisik
                  "laterality_final", "flipped"]  # v4: diagnostik kanonikalisasi lateralitas

running_bytes = 0
if prior_manifest_path is not None:
    tlog(f"Checkpoint ketemu: {prior_manifest_path}")
    prior_df = pd.read_csv(prior_manifest_path)
    done_keys = set(zip(prior_df["StudyInstanceUID"], prior_df["SeriesInstanceUID"]))
    prior_df[MANIFEST_COLS].to_csv(MANIFEST_PATH, index=False)
    running_bytes = int(prior_df["bytes"].fillna(0).sum())
    tlog(f"Series sudah selesai sebelumnya: {len(done_keys)}, {running_bytes/1e9:.2f} GB")
else:
    pd.DataFrame(columns=MANIFEST_COLS).to_csv(MANIFEST_PATH, index=False)

# urutan: semua series studi gold-label dulu, baru sisanya urut StudyInstanceUID (deterministik)
gold_series = train_series[train_series["StudyInstanceUID"].isin(gold_studies)]
rest_series = train_series[~train_series["StudyInstanceUID"].isin(gold_studies)].sort_values(["StudyInstanceUID", "SeriesInstanceUID"])
ordered = pd.concat([gold_series, rest_series]).reset_index(drop=True)

keys = list(zip(ordered["StudyInstanceUID"], ordered["SeriesInstanceUID"]))
todo = ordered[[k not in done_keys for k in keys]].reset_index(drop=True)
if DEBUG_LIMIT_SERIES:
    todo = todo.head(DEBUG_LIMIT_SERIES)

tlog(f"Total series: {len(train_series)} | sudah selesai: {len(done_keys)} | akan diproses run ini (maks): {len(todo)}")
work_items = list(zip(todo["StudyInstanceUID"], todo["SeriesInstanceUID"], todo["Anatomical_Plane"],
                       todo["Fluid_Sensitive"], todo["Fat_Suppression"]))


In [ ]:
error_rows = []
n_done_this_run = 0
hit_cutoff = None

if work_items:
    loop_t0 = time.time()
    with Pool(N_WORKERS) as pool:
        for batch_start in range(0, len(work_items), BATCH_SIZE):
            elapsed = time.time() - T0
            if elapsed > SAFETY_CUTOFF_SECONDS:
                hit_cutoff = "waktu"; break
            if running_bytes > SAFETY_CUTOFF_BYTES:
                hit_cutoff = "ukuran output"; break

            batch = work_items[batch_start: batch_start + BATCH_SIZE]
            results = pool.map(process_one_series, batch)

            rows = []
            for r in results:
                rows.append({"StudyInstanceUID": r["study"], "SeriesInstanceUID": r["series"],
                             "Anatomical_Plane": None, "Fluid_Sensitive": None, "Fat_Suppression": None,
                             "Laterality": r.get("laterality"), "n_instances_total": r.get("n_instances", 0),
                             "n_slices_cached": r.get("n_cached", 0), "status": r["status"],
                             "cache_path": r.get("cache_path"), "bytes": r.get("bytes", 0),
                             "pos_source": r.get("pos_source"), "n_pos_fallback": r.get("n_pos_fallback", 0),
                             "laterality_final": r.get("laterality_final"), "flipped": r.get("flipped", False)})
                running_bytes += r.get("bytes", 0) or 0
                if r.get("error"):
                    error_rows.append({"StudyInstanceUID": r["study"], "SeriesInstanceUID": r["series"], "error": r["error"]})
                n_done_this_run += 1

            batch_df = pd.DataFrame(rows)
            meta = todo[["StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]]
            batch_df = batch_df.drop(columns=["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]).merge(
                meta, on=["StudyInstanceUID", "SeriesInstanceUID"], how="left")
            batch_df[MANIFEST_COLS].to_csv(MANIFEST_PATH, mode="a", header=False, index=False)

            elapsed = time.time() - T0
            loop_elapsed = time.time() - loop_t0
            rate = n_done_this_run / max(1e-6, loop_elapsed)
            n_total_done = len(done_keys) + n_done_this_run
            tlog(f"batch {batch_start//BATCH_SIZE+1}: {n_done_this_run}/{len(work_items)} run ini "
                 f"({n_total_done}/{len(train_series)} total = {n_total_done/len(train_series)*100:.1f}%), "
                 f"{running_bytes/1e9:.2f} GB terpakai, {rate*60:.1f} series/menit")
else:
    tlog("Tidak ada series tersisa.")

if error_rows:
    pd.DataFrame(error_rows).to_csv(OUT_DIR / "preprocessing_errors.csv", mode="a",
                                     header=not (OUT_DIR / "preprocessing_errors.csv").exists(), index=False)

n_total_done = len(done_keys) + n_done_this_run
is_complete = (n_total_done >= len(train_series)) and hit_cutoff is None
print(f"\n{'='*60}\nSTATUS: {'SELESAI PENUH' if is_complete else f'BERHENTI ({hit_cutoff or "belum selesai"})'}")
print(f"Progress: {n_total_done} / {len(train_series)} series ({n_total_done/len(train_series)*100:.1f}%)")
print(f"Output: {running_bytes/1e9:.2f} GB")
print(f"{'='*60}")
if not is_complete:
    print(">>> UNTUK LANJUTKAN: tarik series_manifest.csv, jadikan 'prior_series_manifest.csv', push ulang.")


## 6. Ringkasan akhir

In [ ]:
final_manifest = pd.read_csv(MANIFEST_PATH)
print("Status distribusi:")
print(final_manifest["status"].value_counts())
print()
print("Laterality distribusi (dari series yang berhasil):")
print(final_manifest["Laterality"].value_counts(dropna=False).head(10))
print()
print(f"Total ukuran cache: {final_manifest['bytes'].fillna(0).sum()/1e9:.2f} GB")
print(f"Total series ter-cache OK: {(final_manifest['status']=='OK').sum()} / {len(train_series)}")

print()
print("v2 -- distribusi sumber posisi fisik (diagnostik urutan slice):")
print(final_manifest["pos_source"].value_counts(dropna=False))
print(f"Rata-rata fallback per series (n_pos_fallback): {final_manifest['n_pos_fallback'].mean():.2f}")
print(">>> kalau 'ipp' dominan: fix urutan fisik berhasil terpakai. Kalau 'filename_order' dominan:")
print(">>> tag ImagePositionPatient/ImageOrientationPatient banyak hilang di dataset ini, fallback")
print(">>> ke perilaku v1 -- perlu investigasi tag DICOM lebih lanjut sebelum klaim v2 efektif.")

print()
print("v4 -- distribusi lateralitas final & flip (diagnostik canonicalization, slot Coronal/Axial saja):")
coax = final_manifest[final_manifest["Anatomical_Plane"].isin(["Coronal", "Axial"])]
print(coax["laterality_final"].value_counts(dropna=False))
print(f"Proporsi di-flip (studi KIRI -> dikanonikalisasi): {coax['flipped'].mean()*100:.1f}%")
print(">>> proporsi flip idealnya ~seperempat dari total Coronal+Axial (mengikuti distribusi ~L/R")
print(">>> yg sudah diketahui dari EDA sebelumnya: R=24.8%, L=24.4%, kosong=50.7%). Kalau None")
print(">>> (tidak bisa ditentukan) dominan, cek ulang GEOM_LAT_TRUSTED di §1 -- kemungkinan sinyal")
print(">>> geometris tidak terpercaya di sampel itu.")


## Catatan

- **Reusable di notebook lain**: attach output ini sebagai Kaggle Dataset (`dataset_sources`), lalu baca `study_manifest.csv` + `series_manifest.csv` untuk join label/metadata, dan `np.load(cache_path)['slices']` untuk array (N,224,224) uint8 siap pakai.
- Kalau status bukan "SELESAI PENUH": tarik `series_manifest.csv`, taruh sebagai `prior_series_manifest.csv` di folder kernel ini, push ulang -- series yang sudah selesai otomatis di-skip (pola sama seperti audit exhaustive).